# Shared feature functions

The unchanged feature calculations from notebook 05. Loading this notebook defines functions only; it never rebuilds training features.

In [ ]:
import numpy as np
import pandas as pd
import shapely

metadata_columns = ["trip_id", "group_id", "point_id", "segment_id", "label"]
stationary_mps = 0.5

In [ ]:
def motion(gps_xy, time):
    gaps = np.r_[np.nan, np.diff(time)]
    gaps[gaps <= 0] = np.nan
    speed = np.r_[np.nan, np.linalg.norm(np.diff(gps_xy, axis=0), axis=1)] / gaps
    acceleration = np.r_[np.nan, np.diff(speed)] / gaps
    heading = np.full(len(time), np.nan)
    delta = gps_xy[2:] - gps_xy[:-2]
    heading[1:-1] = np.degrees(np.arctan2(delta[:, 0], delta[:, 1])) % 360
    unreliable = ~np.isfinite(speed) | (speed < stationary_mps)
    unreliable[1:-1] |= (np.linalg.norm(delta, axis=1) < 1) | ~np.isfinite(gaps[2:])
    heading[unreliable] = np.nan
    return gaps, speed, acceleration, heading


def angle_difference(a, b):
    return np.abs((a - b + 180) % 360 - 180)

In [ ]:
def add_features(candidates, nodes_xy, arc_nodes, gps_xy, time):
    features = candidates.sort_values(["group_id", "distance", "segment_id"]).reset_index(drop=True).copy()
    p = features["point_id"].to_numpy()
    s = features["segment_id"].to_numpy()
    start, end = arc_nodes.T
    segments = shapely.linestrings(nodes_xy[arc_nodes])
    gps = shapely.points(gps_xy)
    candidate_segments = segments[s]
    length = shapely.length(segments)
    delta = nodes_xy[end] - nodes_xy[start]
    segment_heading = np.degrees(np.arctan2(delta[:, 0], delta[:, 1])) % 360
    segment_heading[length == 0] = np.nan
    gaps, speed, acceleration, heading = motion(gps_xy, time)

    np.testing.assert_allclose(shapely.distance(candidate_segments, gps[p]), features["distance"], atol=1e-6, rtol=0)
    along = shapely.line_locate_point(candidate_segments, gps[p])
    features["segment_fraction"] = np.divide(along, length[s], out=np.full(len(p), np.nan), where=length[s] > 0)
    features["segment_length_m"] = length[s]
    features["heading_diff_deg"] = angle_difference(heading[p], segment_heading[s])
    features["speed_mps"] = speed[p]
    features["acceleration_mps2"] = acceleration[p]
    features["time_gap_s"] = gaps[p]

    groups = features.groupby("group_id", sort=False)
    features["distance_rank"] = groups.cumcount() + 1
    features["distance_above_nearest_m"] = features["distance"] - groups["distance"].transform("min")
    features["candidate_count"] = groups["segment_id"].transform("size")
    features["heading_diff_rank"] = groups["heading_diff_deg"].rank(method="min")

    for shift, name in [(-2, "prev2"), (-1, "prev"), (1, "next"), (2, "next2")]:
        neighbour = p + shift
        valid = (neighbour >= 0) & (neighbour < len(time))
        distances = np.full(len(p), np.nan)
        differences = np.full(len(p), np.nan)
        distances[valid] = shapely.distance(candidate_segments[valid], gps[neighbour[valid]])
        differences[valid] = angle_difference(heading[neighbour[valid]], segment_heading[s[valid]])
        features[f"distance_{name}_m"] = distances
        features[f"heading_diff_{name}_deg"] = differences

    nearest = np.full(len(time), -1, dtype=np.int64)
    first = features.drop_duplicates("point_id")
    nearest[first["point_id"]] = first["segment_id"]
    for shift, name in [(-1, "prev"), (1, "next")]:
        neighbour = p + shift
        valid = (neighbour >= 0) & (neighbour < len(time))
        road = np.full(len(p), -1, dtype=np.int64)
        road[valid] = nearest[neighbour[valid]]
        valid &= road >= 0
        connected = np.full(len(p), np.nan)
        if shift == -1:
            touches = end[road[valid]] == start[s[valid]]
        else:
            touches = end[s[valid]] == start[road[valid]]
        connected[valid] = (road[valid] == s[valid]) | touches
        features[f"connects_{name}"] = connected

    degree = np.bincount(arc_nodes.ravel(), minlength=len(nodes_xy))
    degree -= np.bincount(start[start == end], minlength=len(nodes_xy))
    features["start_degree"] = degree[start[s]]
    features["end_degree"] = degree[end[s]]
    columns = [c for c in features.columns if c not in metadata_columns]
    features[columns] = features[columns].astype("float32")
    return features